# Week 5 Lab 4: Decision Trees (Titanic Logic & Pipelines)

**Goal**: Predict Titanic survival using a Decision Tree to automatically find the optimal "questions" to ask.

> **Why this lab matters**:
> Unlike Regression models which calculate formulas, Decision Trees build highly interpretable logical flowcharts. They teach us how AI can automatically segment data by reducing **Entropy** (messiness).

> **Structure**:
> We follow the **6-Phase Professional Workflow** and use `OrdinalEncoder` instead of `OneHotEncoder`. We will visualize "The Mind of the Machine" and use **Cross-Validation** to test node stability.

---
## Foreword
Decision Trees are highly interpretable. In this lab, we use the **Titanic Dataset** to see the logic.

1. **Phase 1: Splitting**
2. **Phase 2: Preprocessing (OrdinalEncoding)**
3. **Phase 3: Assembly (Pipeline)**
4. **Phase 4: Training**
5. **Phase 5: Evaluation (Visualization & Cross-Validation)**
6. **Phase 6: Optimization**

### 1.1 Import Dependencies & Load Data
**Concept**: We fetch the Titanic data, extracting `Pclass`, `Sex`, and `Age`.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
import matplotlib.pyplot as plt
import numpy as np

# Load Titanic Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df = df[['Survived', 'Pclass', 'Sex', 'Age']].dropna()

X = df[['Pclass', 'Sex', 'Age']]
y = df['Survived']

---
### 1.2 Phase 1: Data Splitting
**Concept**: We secure 20% of the dataset for an unbiased evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
#### Theory: Ordinal Encoder vs One-Hot Encoder
Decision trees don't calculate slopes, they split data based on thresholds (e.g., `Sex <= 0.5`). Because of this, it is perfectly fine to map text to a single column of numbers (0, 1, 2) rather than creating separate columns.

| Component | Target | Function |
| :--- | :--- | :--- |
| `OrdinalEncoder()` | `['Sex']` | Converts 'female' to 0.0 and 'male' to 1.0. Keeps data sparse and efficient for trees. |
| `remainder='passthrough'` | `Age, Pclass`| Tells the ColumnTransformer to ignore everything else but KEEP IT in the matrix! |

### 1.3 Phase 2 & 3: Preprocessing & Assembly
**Concept**: We only need to convert text to numbers; trees don't care about standardization (Scaling).
**Solution**: We define the `OrdinalEncoder` in `ColumnTransformer` and build a pipeline with a max depth of 3.


In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(), ['Sex'])
], remainder='passthrough')

workflow = Pipeline([
    ('pre', preprocessor),
    ('model', tree.DecisionTreeClassifier(criterion='entropy', max_depth=3))
])

---
### 1.4 Phase 4: Training
**Concept**: The tree calculates the "Information Gain" (reduction in Entropy) for every possible feature and threshold, and creates a split.
**Solution**: Call `.fit()` on the pipeline.


In [ ]:
workflow.fit(X_train, y_train)
print("Pipeline trained successfully.")

---
### 1.5 Phase 5: Evaluation (Visualization)
**Concept**: One of the biggest strengths of Decision Trees is that we can literally view the mathematical conditions.
**Solution**: We use `tree.plot_tree` to render the model's logic.


In [ ]:
clf_model = workflow.named_steps['model']
# Get transformed feature names from the preprocessor
feature_names = workflow.named_steps['pre'].get_feature_names_out()
feature_names = [name.split('__')[1] for name in feature_names] # Clean names

plt.figure(figsize=(15, 10))
tree.plot_tree(clf_model, 
               feature_names=feature_names, 
               class_names=['Perished', 'Survived'], 
               filled=True, 
               rounded=True)
plt.title("Titanic Decision Tree: Reducing Entropy")
plt.show()

---
### 1.6 Phase 5: Cross-Validation
**Concept**: A major flaw of Decision Trees is **High Variance**. Small changes in the training data can result in a completely different tree structure.
**Solution**: We use Cross-Validation to see how much the accuracy jumps around when the data changes.


In [ ]:
scores = cross_val_score(workflow, X, y, scoring='accuracy', cv=5)
print("Accuracy Scores across 5 folds:", np.round(scores, 3))
print(f"\nAverage CV Accuracy: {scores.mean():.2%}")
print(f"Standard Deviation: {scores.std():.2%} (The higher this is, the higher the variance!)")

> **Observation**: Notice the standard deviation. Decision trees tend to fluctuate slightly more than Logistic Regression because they overfit to specific subsets of data. This variance is exactly what Random Forests (Lab 5) are designed to fix.

**Task 1**: In Phase 3, change `max_depth=3` to `max_depth=None`. Rerun the entire notebook. What happens to the visualization? What happens to the Cross-Validation average accuracy and standard deviation?

<details>
<summary><strong> Click here for Solution (Try it yourself first!)</strong></summary>

If you remove the depth limit, the tree becomes immense and highly complex. 

The Average CV Accuracy will **DROP** and the standard deviation will **INCREASE**, proving that an infinitely deep tree simply **Overfits** to noise rather than learning genuine patterns.
</details>

---
### Summary
Look at the first split (The Root Node). Was it `Sex`? On the Titanic, "Women and children first" was a strong rule — the Decision Tree discovered this automatically by calculating which question reduced **Entropy** (messiness) the most! We achieved this professionally using pipelines.